In [48]:
import torch
from torch import nn
from torch.nn import functional as F


class Resblock(nn.Module):
    def __init__(self, input_channels, output_channels, use_1con=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, kernel_size=3, padding=1, stride=stride)
        self.conv2 = nn.Conv2d(output_channels, output_channels, kernel_size=3, padding=1)
        if use_1con:
            self.conv3 = nn.Conv2d(input_channels, output_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.bn2 = nn.BatchNorm2d(output_channels)
    
    def forward(self, X):
        Y = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(X)))))
        if self.conv3:
            X = self.conv3(X)
        Y = Y + X
        return F.relu(Y)

In [49]:
blk = Resblock(3,3)
X = torch.rand([4,3,6,6])
Y = blk(X)
Y.shape

torch.Size([4, 3, 6, 6])

In [50]:
blk2 = Resblock(3,6,True)
X = blk2(X)
X.shape

torch.Size([4, 6, 6, 6])

In [51]:
b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),    # 1,64,112,112
                   nn.BatchNorm2d(64), nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))        # 1,64,56,56

def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Resblock(input_channels, num_channels,
                                use_1con=True, stride=2))
        else:
            blk.append(Resblock(num_channels, num_channels))
    return blk


b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))              # 1,64,56,56 
b3 = nn.Sequential(*resnet_block(64, 128, 2))                               # 1,128,28,28
b4 = nn.Sequential(*resnet_block(128, 256, 2))                              # 1,256,14,14
b5 = nn.Sequential(*resnet_block(256, 512, 2))                              # 1,512,7,7

net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),                            # 1,512,1,1
                    nn.Flatten(), nn.Linear(512, 10))                       # 1,512 --> 1,10


X = torch.rand(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 10])
